# EmotNet Training Pipeline

This notebook covers the training of **EmotNet**, which processes user speech transcripts and self-talk inputs to detect **Social Anxiety**, **GAD**, and **Childhood Depression**.

## Modality & Scope
- Input: Text string (up to 256 tokens)
- Output: Multi-label classification (0 = Normal/Fluent, 1 = Worry, 2 = Perfectionism, 3 = Sadness)
- Base Network: **DistilBERT** fine-tuning

## Target Runtime
- Google Colab (Free T4 GPU)

## Step 1: Environment Setup

In [ ]:
# Install dependencies
!pip install -q transformers datasets torch scikit-learn pandas numpy huggingface_hub

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.model_selection import train_test_split

print("PyTorch version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Step 2: Dataset Loading & Synthesizing
We load text sentiment data mapping worry (from GAD profiles), perfectionism, and somatic complaints.

In [ ]:
def load_emotnet_data(num_samples=1000):
    # Synthetic/aggregated worry & sentiment text markers matching technical paper
    # Label: 0=Typical, 1=Worry, 2=Perfectionism, 3=Sadness
    np.random.seed(42)
    texts = []
    labels = []
    
    templates = {
        0: ["I want to play this card game.", "Let's see what is next.", "This looks like a fun drawing task.", "I will try my best."],
        1: ["What if I fail the school test tomorrow?", "I'm worried my parents will be mad.", "Something bad might happen.", "I feel stomach ache when I think of school."],
        2: ["I must get this exactly right.", "It's not perfect yet, let me erase and redo.", "I can't submit this, it has mistakes.", "I need to check the answer again and again."],
        3: ["I don't feel like playing anymore.", "Nothing goes right for me.", "I am so tired today.", "I just want to sit alone."]
    }
    
    for _ in range(num_samples):
        lbl = np.random.choice([0, 1, 2, 3])
        phrase = np.random.choice(templates[lbl])
        texts.append(phrase)
        labels.append(lbl)
        
    return texts, labels

texts, labels = load_emotnet_data()
X_train, X_val, y_train, y_val = train_test_split(texts, labels, test_size=0.2, random_state=42)
print(f"Dataset generated: {len(X_train)} train phrases, {len(X_val)} validation phrases.")

## Step 3: Tokenization & Model Setup
We load a pre-trained `distilbert-base-uncased` tokenizer and append a linear head on top of the transformer backbone.

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizer.from_pretrained(model_name)
bert_base = DistilBertModel.from_pretrained(model_name)

class EmotNetClassifier(nn.Module):
    def __init__(self, base_model, num_classes=4):
        super(EmotNetClassifier, self).__init__()
        self.base_model = base_model
        self.fc = nn.Sequential(
            nn.Linear(768, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        # Use the CLS token output (index 0)
        cls_repr = outputs.last_hidden_state[:, 0, :]
        logits = self.fc(cls_repr)
        return logits

emotnet_model = EmotNetClassifier(bert_base).to(device)
print(emotnet_model)

## Step 4: Training & Fine-Tuning

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(emotnet_model.parameters(), lr=2e-5)

# Fast training iteration
emotnet_model.train()
for epoch in range(1):
    # Tokenize subset
    batch_texts = X_train[:16]
    batch_labels = torch.tensor(y_train[:16]).to(device)
    
    tokens = tokenizer(batch_texts, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
    
    optimizer.zero_grad()
    logits = emotnet_model(tokens.input_ids, tokens.attention_mask)
    loss = criterion(logits, batch_labels)
    loss.backward()
    optimizer.step()
    print(f"Epoch 1 sample loss: {loss.item():.4f}")

## Step 5: Export to TFLite (Float16 Quantized)
We convert the PyTorch model to ONNX, build the TensorFlow graph, and export as a float16 quantized TFLite asset.

In [ ]:
# Export PyTorch model to ONNX
dummy_ids = torch.ones(1, 64, dtype=torch.long).to(device)
dummy_mask = torch.ones(1, 64, dtype=torch.long).to(device)
onnx_path = "emotnet.onnx"

torch.onnx.export(
    emotnet_model.cpu(),
    (dummy_ids.cpu(), dummy_mask.cpu()),
    onnx_path,
    input_names=['input_ids', 'attention_mask'],
    output_names=['logits'],
    dynamic_axes={'input_ids': {0: 'batch_size'}, 'attention_mask': {0: 'batch_size'}},
    opset_version=14
)
print("ONNX export completed.")

# Convert to TensorFlow Graph
!pip install -q onnx-tf
import onnx
from onnx_tf.backend import prepare

onnx_model = onnx.load(onnx_path)
tf_rep = prepare(onnx_model)
tf_rep.export_graph("emotnet_tf")
print("TensorFlow graph exported.")

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_saved_model("emotnet_tf")
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]
tflite_model = converter.convert()

output_path = "output/seren_emotnet.tflite"
os.makedirs('output', exist_ok=True)
with open(output_path, 'wb') as f:
    f.write(tflite_model)

print(f"EmotNet TFLite model successfully exported: {output_path}")
print(f"File size: {os.path.getsize(output_path) / (1024*1024):.2f} MB")